# YOLOv8 完整演示 Notebook

本 Notebook 展示 YOLOv8 项目的主要功能：
1. 基础推理
2. 插件系统使用
3. 模型训练
4. 模型评估
5. 模型导出
6. 可视化

## 0. 环境准备

In [ ]:
import sys
from pathlib import Path

# 确保项目根目录在路径中
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. 基础推理

In [ ]:
# 加载模型
model = YOLO('yolov8n.pt')

# 推理
results = model('https://ultralytics.com/images/bus.jpg')

# 显示结果
for r in results:
    im_array = r.plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
    
    # 打印检测信息
    boxes = r.boxes
    if boxes is not None:
        print(f"检测到 {len(boxes)} 个目标:")
        for box in boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            name = r.names[cls_id]
            print(f"  - {name}: {conf:.2f}")

## 2. 插件系统

In [ ]:
from models.registry import list_plugins, get_plugin

# 列出所有可用插件
plugins = list_plugins()
for category, names in plugins.items():
    print(f"\n{'='*50}")
    print(f"【{category}】({len(names)} 个)")
    print('='*50)
    for name in names:
        try:
            meta = get_plugin(name).__doc__ or '(无描述)'
            print(f"  • {name}: {meta.strip().split(chr(10))[0]}")
        except:
            print(f"  • {name}")

In [ ]:
# 测试插件实例化
from models.registry import PLUGIN_REGISTRY

# 测试 SE 注意力
se = PLUGIN_REGISTRY.build('se_attention', in_channels=64, reduction=16)
x = torch.randn(1, 64, 32, 32)
y = se(x)
print(f"SE Attention: 输入 {x.shape} → 输出 {y.shape}")
print(f"参数量: {sum(p.numel() for p in se.parameters()):,}")

In [ ]:
# 从配置文件构建模型
from models.plugin_builder import build_model

# 使用带有注意力插件的配置
config = {
    "model": {
        "base": "yolov8n.pt",
        "plugins": {
            "backbone": [
                {"type": "se_attention", "params": {"in_channels": 64, "reduction": 8}}
            ]
        }
    }
}

model_with_plugins = build_model(config_dict=config)
print(f"模型构建成功: {type(model_with_plugins)}")

## 3. 训练

In [ ]:
# 快速训练演示（2 轮，仅验证流程）
model = YOLO('yolov8n.pt')

results = model.train(
    data='coco128.yaml',
    epochs=2,
    imgsz=320,
    batch=8,
    device='cpu',
    verbose=True,
)

print(f"训练完成！模型保存在: {results.save_dir}")

## 4. 指标计算

In [ ]:
from utils.metrics import bbox_iou

# 测试 IoU 计算
box1 = torch.tensor([[0, 0, 100, 100], [50, 50, 150, 150]], dtype=torch.float32)
box2 = torch.tensor([[30, 30, 130, 130]], dtype=torch.float32)

iou = bbox_iou(box1.numpy(), box2.numpy())
print(f"IoU 矩阵:\n{iou}")

ciou = bbox_iou(box1.numpy(), box2.numpy(), mode='ciou')
print(f"CIoU 矩阵:\n{ciou}")

## 5. 可视化

In [ ]:
from utils.plots import draw_detections

# 创建测试图像
img = np.zeros((480, 640, 3), dtype=np.uint8)
img[:] = (180, 180, 180)

# 模拟检测结果
boxes = np.array([[100, 100, 300, 400], [400, 200, 550, 350]])
classes = np.array([0, 2])
confs = np.array([0.95, 0.72])
names = {0: 'person', 2: 'car'}

result = draw_detections(img, boxes, classes, confs, names)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Detection Demo')
plt.show()

## 6. 模型导出

In [ ]:
# 导出为 ONNX
model = YOLO('yolov8n.pt')

try:
    onnx_path = model.export(format='onnx', imgsz=320, simplify=True)
    print(f"ONNX 导出成功: {onnx_path}")
    print(f"文件大小: {Path(onnx_path).stat().st_size / 1024:.1f} KB")
except Exception as e:
    print(f"导出失败（可能需要 onnx 包）: {e}")

## 总结

本 Notebook 展示了 YOLOv8 项目的核心功能：
- ✅ 模型加载与推理
- ✅ 插件系统注册与使用
- ✅ 模型训练
- ✅ 指标计算
- ✅ 结果可视化
- ✅ 模型导出

更多功能请参考 `scripts/` 目录下的命令行工具和 `tutorials/plugin_guide.md`。